[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a1cd/Shears/blob/main/Notebooks/Shears.ipynb)

# Shears: prune and heal a Mixture-of-Experts model

> ## Read this first
>
> **The real target is MLX on an Apple Silicon Mac, with Gemma 4 26B-A4B.** Every number published for Shears (speed, memory, quality, energy) was measured there.
>
> **Two modes, chosen in the first code cell.** `MODEL = "tiny"` prunes and heals a random 4-layer model so you can watch the whole pipeline work (it writes gibberish before and after; it is not a way to get a good pruned model). `MODEL = "gemma"` (the default) downloads the real `google/gemma-4-26B-A4B-it` (about 52 GB), prunes it to 8 experts per layer, heals it for a handful of steps per block and exports it, then generates from the export. Neither is a benchmark: do not compare timings or losses with the Mac numbers, because Linux has no Metal, no zero-copy memory mapping and no unified memory.
>
> **The gemma mode on Colab depends on a CUDA binary that has never run on a GPU.** On Linux x86_64 with an NVIDIA GPU of compute capability 7.5 or newer the notebook downloads a CUDA build of `shears-cli` that CI compiles on a machine with no GPU. Nobody has run it on a GPU yet; this notebook, opened on a Colab GPU runtime, is its first test. If that binary will not start, the notebook prints why and falls back to the CPU build, and then **stops before the 52 GB download**, because the real model on a CPU would take far too long (a setting lets you go on anyway). The CUDA build is also unmeasured: no speed or memory figure for it exists. The tiny mode never needs a GPU, and CI runs it on every change on the CPU build.
>
> If something here fails or looks slow, that says nothing about Shears on a Mac.

Shears prunes a Mixture-of-Experts model (removes experts) and heals it (retrains what is left, with small adapters, against the original model). It is a Swift library on [mlx-swift](https://github.com/ml-explore/mlx-swift). Python is only glue in this notebook: it finds your platform, runs shell commands and, at the end, loads the result with Hugging Face transformers.

**What runs, in order**

1. Pick `MODEL` (next cell) and detect the platform (Apple Silicon Mac, or Linux such as Colab) and any NVIDIA GPU.
2. Set up `shears-cli`. On a Mac that is a build with `xcodebuild`. On x86_64 Linux (Colab) it downloads a prebuilt binary that CI builds from this repository and checks its SHA-256; nothing is built. With a suitable NVIDIA GPU that is the CUDA build, otherwise the CPU build. The notebook then prints which device MLX uses. (Building from source on Linux is an optional cell at the very end, for changing Shears itself.)
3. `tiny`: run the pipeline on a tiny random Gemma 4 MoE: `select` (choose which experts to keep), `prepass` (run the original model once and store its outputs), `heal` (train the adapters, three stages), `export` (merge them, write only the kept experts, Hugging Face layout, bf16). `gemma`: check the disk, download the model, then `run` (the block-sequential pipeline: select and heal one block of layers at a time, streaming the weights) and `export`.
4. Load that export with transformers and generate: a few tokens of gibberish from the tiny model, or an answer to a real prompt from the pruned Gemma.

**Time.** `tiny`: on a Mac (M3 Max) a cold setup takes about 100 seconds once Xcode is ready, and the pipeline 2 to 15 seconds; on Linux the setup is a download of a few tens of megabytes (CI measures it on a plain x86_64 runner; **not yet measured on Colab**). `gemma`: the download is about 52 GB (roughly 10 to 20 minutes at typical Colab speeds, the notebook prints its own estimate), and the run itself has **never been timed on a GPU or on Colab**; expect an hour or more on a T4. Section 5 prints the time of each phase. Building the binary yourself instead (the optional cell at the end) takes on the order of 15 to 30 minutes on a Colab CPU runtime (an estimate, **not measured on Colab**).

**On Colab** choose a GPU runtime for `gemma` (Runtime, Change runtime type; a T4 is the smallest the CUDA build targets) and a CPU runtime is fine for `tiny`. **Try `tiny` on a GPU runtime first:** it already uses the CUDA binary when there is a GPU, so it is a two-minute first test of that binary with no 52 GB download. Run the cells top to bottom; each shell step runs in its own `bash`, with `PATH` set from Python.

## Choose what to run

In [ ]:
import os

# "gemma" (default): the real google/gemma-4-26B-A4B-it. Downloads about 52 GB, prunes it to 8 experts per layer, heals every block
#                    for a few steps and exports it; on a Colab T4 expect roughly an hour or more (never timed on a GPU).
# "tiny":            a random 4-layer model made on the spot. Nothing to download; under 2 minutes, on any platform, no GPU needed.
MODEL = "gemma"

# CI sets SHEARS_MODEL, or just runs on GitHub Actions (GITHUB_ACTIONS is set there), and always gets the tiny run.
if os.environ.get("SHEARS_MODEL"):
    MODEL = os.environ["SHEARS_MODEL"]
elif os.environ.get("GITHUB_ACTIONS") == "true":
    MODEL = "tiny"
assert MODEL in ("gemma", "tiny"), f'MODEL must be "gemma" or "tiny", not {MODEL!r}'
print("MODEL =", MODEL)

## 1. Platform and settings

In [ ]:
import os, sys, platform, shutil, subprocess, time, json, pathlib

# --- Settings ---------------------------------------------------------------------------------
SHEARS_GIT_URL = os.environ.get("SHEARS_GIT_URL", "https://github.com/a1cd/Shears.git")   # used only when this notebook is not inside a Shears checkout
SHEARS_REF = os.environ.get("SHEARS_REF", "main")            # branch or tag to build in that case
SWIFT_VERSION = "6.3.3"        # Linux only; the version the Linux build was verified with
TRANSFORMERS_VERSION = "5.20.2"  # pinned. Gemma 4 first appears in transformers 5.5.0; 5.20.2 is the version tested here
RUN_EXPERIMENTAL_CUDA = False  # Linux + NVIDIA GPU only. UNTESTED. Only used by the optional build-from-source cell at the end; read it before turning this on.
# Linux x86_64: the notebook downloads this prebuilt shears-cli (CPU) and checks its SHA-256, which is read from PREBUILT_URL + ".sha256"
# unless you pin it in PREBUILT_SHA256. A local file works too (a path or a file:// URL). If the download fails the notebook stops;
# the optional last cell builds from source instead.
PREBUILT_URL = os.environ.get("SHEARS_PREBUILT_URL", "https://github.com/a1cd/Shears/releases/download/linux-x86_64-latest/shears-cli-linux-x86_64.tar.gz")
PREBUILT_SHA256 = os.environ.get("SHEARS_PREBUILT_SHA256", "")
# Linux x86_64 with an NVIDIA GPU (compute capability 7.5 or newer): the notebook first tries this CUDA build, EXPERIMENTAL and never run on a
# GPU before this notebook. It needs the CUDA libraries a Colab GPU runtime already has (they are not in the tarball). If it will not start,
# the notebook prints why and uses the CPU build above. SHEARS_USE_CUDA=0 skips it.
CUDA_PREBUILT_URL = os.environ.get("SHEARS_CUDA_PREBUILT_URL", "https://github.com/a1cd/Shears/releases/download/linux-x86_64-cuda-experimental/shears-cli-linux-x86_64-cuda.tar.gz")
CUDA_PREBUILT_SHA256 = os.environ.get("SHEARS_CUDA_PREBUILT_SHA256", "")
USE_CUDA_BINARY = os.environ.get("SHEARS_USE_CUDA", "1") != "0"
CCCL_VERSION = "3.1.3"        # the CCCL headers MLX was built with; MLX compiles some kernels at run time against them (CUDA path only)
# MODEL = "gemma":
GEMMA_REPO = "google/gemma-4-26B-A4B-it"
GEMMA_STEPS_PER_BLOCK = 10     # healing steps for each of the 5 blocks of layers (export needs adapters for every layer, so every block is healed)
ALLOW_GEMMA_ON_CPU = False     # if MLX ends up on the CPU (no usable GPU) the notebook stops before the 52 GB download; True goes on anyway

# --- Platform ---------------------------------------------------------------------------------
IS_MAC = sys.platform == "darwin"
IS_LINUX = sys.platform.startswith("linux")
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
ARCH = platform.machine()
if IS_MAC and ARCH != "arm64":
    raise SystemExit("Shears needs Apple Silicon (MLX does not run on Intel Macs). Use Colab or Linux instead.")
if not (IS_MAC or IS_LINUX):
    raise SystemExit(f"Unsupported platform: {sys.platform}. Use an Apple Silicon Mac or x86_64 Linux.")
print(f"platform: {'Apple Silicon Mac' if IS_MAC else 'Linux'} ({ARCH}){', Colab' if IN_COLAB else ''}, Python {platform.python_version()}")
if IS_LINUX:
    print(open("/etc/os-release").read().split("PRETTY_NAME=")[1].splitlines()[0], "| CPUs:", os.cpu_count())

def detect_nvidia():
    """(info, why_not): info is {"name", "cc", "mem_mib", "driver"} for GPU 0 from nvidia-smi, else None and the reason."""
    exe = shutil.which("nvidia-smi")
    if not exe:
        return None, "nvidia-smi not found: no NVIDIA GPU on this runtime"
    r = subprocess.run([exe, "--query-gpu=name,compute_cap,memory.total,driver_version", "--format=csv,noheader,nounits"],
                       capture_output=True, text=True)
    if r.returncode != 0 or not r.stdout.strip():
        return None, f"nvidia-smi failed (exit {r.returncode}): {(r.stderr or r.stdout).strip()[:200]}"
    f = [x.strip() for x in r.stdout.strip().splitlines()[0].split(",")]
    try:
        return {"name": f[0], "cc": float(f[1]), "mem_mib": int(float(f[2])), "driver": f[3]}, ""
    except (IndexError, ValueError):
        return None, f"could not read nvidia-smi output: {r.stdout.strip()[:200]}"

GPU, GPU_WHY_NOT = detect_nvidia() if IS_LINUX else (None, "not Linux")
if IS_LINUX:
    print(f"NVIDIA GPU: {GPU['name']}, compute capability {GPU['cc']}, {GPU['mem_mib']} MiB, driver {GPU['driver']}" if GPU else f"NVIDIA GPU: none ({GPU_WHY_NOT})")

# --- Where things live ------------------------------------------------------------------------
# SwiftPM names a path dependency after its directory, so the sources must sit in a directory called "Shears".
WORK = pathlib.Path(os.environ.get("SHEARS_WORK") or ("/content/shears-work" if IN_COLAB else pathlib.Path.home() / "shears-notebook-work"))
SRC = WORK / "Shears"
WORK.mkdir(parents=True, exist_ok=True)

# Inside a checkout (Notebooks/ next to Package.swift)? Use that code. Otherwise clone it.
here = pathlib.Path.cwd()
LOCAL_REPO = next((p for p in (here.parent, here) if (p / "Package.swift").exists() and (p / "Sources" / "Shears").is_dir()), None)

# --- Helpers ----------------------------------------------------------------------------------
PHASES = []   # (name, seconds)

def sh(script, name, cwd=None, env=None, quiet=False):
    """Run a bash script (stops at the first error), stream its output, and record how long it took.
    quiet=True prints only the last few lines, or the last 60 if it fails (for noisy Xcode builds)."""
    e = dict(os.environ, **(env or {}))
    t0 = time.time()
    p = subprocess.Popen(["bash", "-c", "set -eo pipefail\n" + script], cwd=cwd, env=e,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        if quiet:
            tail = (tail + [line])[-60:]
        else:
            print(line, end="")
    rc = p.wait()
    dt = time.time() - t0
    PHASES.append((name, dt))
    if quiet:
        print("".join(tail if rc != 0 else []), end="")
    print(f"--- {name}: {dt:.1f} s")
    if rc != 0:
        raise RuntimeError(f"{name} failed with exit status {rc}")

def add_to_path(*dirs):
    os.environ["PATH"] = os.pathsep.join([str(d) for d in dirs] + [os.environ["PATH"]])

def phase_table():
    for n, s in PHASES:
        print(f"{n:42} {s:8.1f} s")
    print(f"{'total':42} {sum(s for _, s in PHASES):8.1f} s")

print("work directory:", WORK)
print("Shears code from:", LOCAL_REPO if LOCAL_REPO else f"{SHEARS_GIT_URL} @ {SHEARS_REF}")

## 2. Setup: get `shears-cli`

First the sources go into `SRC` (a copy of this checkout, or a fresh clone; the pipeline reads its example config). Then `shears-cli` itself: on a Mac it is built with `xcodebuild` (2a); on x86_64 Linux and Colab the prebuilt binary is downloaded (2b). The cell for the other platform prints that it is skipping.

In [ ]:
# Put the sources in WORK/Shears.
if LOCAL_REPO:
    if SRC.exists():
        shutil.rmtree(SRC)
    shutil.copytree(LOCAL_REPO, SRC, ignore=shutil.ignore_patterns(".git", ".build", "Packages", ".swiftpm", "DerivedData", "*.xcodeproj", "__pycache__", ".ipynb_checkpoints"))
    print("copied", LOCAL_REPO, "->", SRC)
elif not (SRC / "Package.swift").exists():
    sh(f'git clone --depth 1 --branch "{SHEARS_REF}" "{SHEARS_GIT_URL}" "{SRC}"', "git clone Shears")
sh("git log -1 --format='sources at %h %s' 2>/dev/null || true", "sources", cwd=SRC)

### 2a. Apple Silicon Mac

Needs Xcode with the Metal Toolchain (`xcodebuild -downloadComponent MetalToolchain` if the build complains about `metal`). Plain `swift build` cannot compile MLX's Metal shaders, so this uses `xcodebuild`.

In [ ]:
if IS_MAC:
    sh("xcrun -f metal >/dev/null 2>&1 || { echo 'No Metal Toolchain. Run: xcodebuild -downloadComponent MetalToolchain'; exit 1; }", "check Metal toolchain")
    sh(f'cd "{SRC}/CLI" && xcodebuild build -scheme shears-cli -destination "platform=macOS" -configuration Release '
       f'-derivedDataPath "{WORK}/derived" -quiet 2>&1', "xcodebuild shears-cli", quiet=True)
    SHEARS_CLI = WORK / "derived/Build/Products/Release/shears-cli"
else:
    print("not a Mac: skipping (the Linux cells below do the setup)")

### 2b. Linux x86_64 and Colab: the prebuilt `shears-cli`

On x86_64 Linux this cell sets up a prebuilt `shears-cli` (Swift runtime linked statically, the few CPU libraries it needs bundled next to it), checks its SHA-256, unpacks it, runs it once as a test, installs `g++` if it is missing (MLX compiles fused CPU kernels with it at run time) and puts the binary on `PATH`. CI builds both from this repository on Ubuntu 24.04; they need glibc 2.39 or newer (Colab's GPU image is Ubuntu 24.04).

- **With an NVIDIA GPU of compute capability 7.5 or newer** (`nvidia-smi` says so) it tries the **CUDA build** first. It is **experimental and has never been run on a GPU**: CI compiles it on a machine that has none. The tarball contains no NVIDIA library; the cell finds the CUDA 13 runtime, cuBLAS, cuFFT, NVRTC and cuDNN libraries that a Colab GPU image has (in `/usr/local/cuda*`, or in the `nvidia-*` pip wheels), fetches the CCCL headers MLX needs for the kernels it compiles at run time, and then runs `shears-cli device`, which does one small matrix product on MLX's default device. If anything in that chain fails it prints the reason and falls back to the CPU build.
- **Otherwise, or after such a failure,** the **CPU build**. Nothing is built behind your back: if even that cannot be set up (no network, wrong checksum, the binary does not run on this system) the cell stops and says why. To build from source instead, use the optional cell at the end of the notebook, then rerun from section 3.

In [ ]:
PREBUILT_OK = False     # a prebuilt shears-cli (CUDA or CPU) was set up
CUDA_OK = False         # ... and it is the CUDA one, and MLX runs a real operation on the GPU with it
CUDA_REASON = None      # why the CUDA build was not used, when it was skipped or failed
if IS_LINUX and ARCH != "x86_64":
    raise RuntimeError(f"There is no prebuilt shears-cli for Linux {ARCH}. Build from source with the optional cell at the end of the notebook.")
if IS_LINUX:
    import hashlib, tarfile, urllib.request, zipfile, glob, re, tempfile

    def fetch(url, dest):
        if "://" not in url:
            url = pathlib.Path(url).resolve().as_uri()
        req = urllib.request.Request(url, headers={"User-Agent": "shears-notebook"})
        with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
            shutil.copyfileobj(r, f)

    def fetch_and_unpack(url, pinned_sha, dest):
        """Download url (and, unless pinned_sha is given, url + '.sha256'), check the SHA-256, unpack into dest, return bin/shears-cli."""
        shutil.rmtree(dest, ignore_errors=True)
        dest.mkdir(parents=True)
        print("downloading", url)
        fetch(url, dest / "shears-cli.tar.gz")
        want = pinned_sha.strip().lower()
        if not want:
            fetch(url + ".sha256", dest / "shears-cli.tar.gz.sha256")
            want = (dest / "shears-cli.tar.gz.sha256").read_text().split()[0].lower()
        h = hashlib.sha256()
        with open(dest / "shears-cli.tar.gz", "rb") as f:
            for block in iter(lambda: f.read(1 << 20), b""):
                h.update(block)
        if h.hexdigest() != want:
            raise ValueError(f"SHA-256 mismatch: expected {want}, got {h.hexdigest()}")
        print("SHA-256 ok:", want, f"({(dest / 'shears-cli.tar.gz').stat().st_size / 1e6:.1f} MB)")
        with tarfile.open(dest / "shears-cli.tar.gz") as tf:
            if hasattr(tarfile, "data_filter"):
                tf.extractall(dest / "pkg", filter="data")
            else:
                tf.extractall(dest / "pkg")
        return next((dest / "pkg").glob("*/bin/shears-cli"))

    def install_gxx():
        # g++ is needed at run time: MLX compiles fused CPU kernels with it.
        sh(r"""
SUDO=""; [ "$(id -u)" = 0 ] || SUDO=sudo
if ! command -v g++ >/dev/null; then
  export DEBIAN_FRONTEND=noninteractive
  $SUDO apt-get update -qq && $SUDO apt-get install -y -qq g++
fi
g++ --version | head -1
""", "g++ for MLX's CPU kernels")

    # ---------------------------------------------------------------------------------------------
    # The CUDA build, if there is a GPU it was built for. Any failure here is a printed reason, then the CPU build.
    # ---------------------------------------------------------------------------------------------
    def nvidia_library_dirs():
        """Directories where the NVIDIA libraries may live on this machine: the toolkit and the driver first, then pip wheels."""
        system = []
        for pat in ["/usr/local/cuda/lib64", "/usr/local/cuda-*/lib64", "/usr/local/cuda-*/targets/x86_64-linux/lib",
                    "/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"]:
            system += glob.glob(pat)
        import site, sysconfig
        wheels = []
        for sp in {*site.getsitepackages(), sysconfig.get_paths()["purelib"]}:
            wheels += sorted({os.path.dirname(p) for p in glob.glob(os.path.join(sp, "nvidia", "**", "lib*.so*"), recursive=True)})
        return list(dict.fromkeys(system)), list(dict.fromkeys(wheels))

    def missing_libraries(cli, ld_path):
        r = subprocess.run(["ldd", str(cli)], capture_output=True, text=True, env=dict(os.environ, LD_LIBRARY_PATH=ld_path))
        return [l.split("=>")[0].strip() for l in r.stdout.splitlines() if "not found" in l]

    def try_cuda_build():
        """Set up the CUDA build and prove it runs an operation on the GPU. Returns the path of shears-cli, or raises with the reason."""
        if not USE_CUDA_BINARY:
            raise RuntimeError("SHEARS_USE_CUDA=0")
        if not GPU:
            raise RuntimeError(GPU_WHY_NOT)
        if GPU["cc"] < 7.5:
            raise RuntimeError(f"{GPU['name']} has compute capability {GPU['cc']}; the CUDA build needs 7.5 or newer (MLX's own minimum)")
        t1 = time.time()
        cli = fetch_and_unpack(CUDA_PREBUILT_URL, CUDA_PREBUILT_SHA256, WORK / "prebuilt-cuda")
        print("CUDA build:", (cli.parent.parent / "BUILD_INFO.txt").read_text().strip().replace("\n", "\n  "))
        # 1. The NVIDIA libraries are not in the tarball. Find them.
        system, wheels = nvidia_library_dirs()
        ld = ":".join(system + [d for d in os.environ.get("LD_LIBRARY_PATH", "").split(":") if d])
        missing = [m for m in missing_libraries(cli, ld) if not m.startswith("libcuda.so")]
        if missing:
            print("libraries not on the system path:", missing, "- looking in pip's nvidia wheels")
            for m in missing:
                hits = [d for d in wheels if glob.glob(os.path.join(d, m))]
                hits.sort(key=lambda d: "cu13" not in d)      # a CUDA 13 wheel before a CUDA 12 one for libraries with the same name
                if hits:
                    ld = hits[0] + ":" + ld
            missing = [m for m in missing_libraries(cli, ld) if not m.startswith("libcuda.so")]
        if missing:
            print("still missing", missing, "- trying pip install of the CUDA 13 runtime wheels")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "nvidia-cuda-runtime",
                            "nvidia-cublas", "nvidia-cuda-nvrtc", "nvidia-cufft", "nvidia-cudnn-cu13"], capture_output=True, text=True)
            system, wheels = nvidia_library_dirs()
            for m in missing:
                hits = sorted([d for d in wheels if glob.glob(os.path.join(d, m))], key=lambda d: "cu13" not in d)
                if hits:
                    ld = hits[0] + ":" + ld
            missing = [m for m in missing_libraries(cli, ld) if not m.startswith("libcuda.so")]
        if missing:
            raise RuntimeError(f"these CUDA libraries are not on this machine: {missing}")
        if "libcuda.so.1" in " ".join(missing_libraries(cli, ld)):
            raise RuntimeError("libcuda.so.1 (the NVIDIA driver library) is not found although nvidia-smi works")
        # 2. MLX compiles some kernels at run time (NVRTC) against CCCL headers it looks for in <bundle>/include/cccl. Fetch the pinned ones.
        try:
            z = WORK / "prebuilt-cuda" / "cccl.zip"
            fetch(f"https://github.com/NVIDIA/cccl/releases/download/v{CCCL_VERSION}/cccl-v{CCCL_VERSION}.zip", z)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(WORK / "prebuilt-cuda" / "cccl")
            root = next(p for p in [*(WORK / "prebuilt-cuda" / "cccl").glob("*/include"), WORK / "prebuilt-cuda" / "cccl" / "include"] if p.is_dir())
            target = cli.parent.parent / "include" / "cccl"
            for sub in ("cuda", "nv"):
                shutil.copytree(root / sub, target / sub, dirs_exist_ok=True)
            print(f"CCCL {CCCL_VERSION} headers for MLX's run-time kernels:", target)
        except Exception as e:
            print(f"WARNING: could not fetch the CCCL headers ({type(e).__name__}: {e}); kernels MLX compiles at run time may fail")
        # 3. Does it run, on the GPU? One real operation, not --help.
        env = dict(os.environ, LD_LIBRARY_PATH=ld)
        r = subprocess.run([str(cli), "device"], capture_output=True, text=True, timeout=900, env=env)
        out = (r.stdout + r.stderr).strip()
        print(out[-1500:])
        if r.returncode != 0:
            raise RuntimeError(f"`shears-cli device` exited {r.returncode}")
        if "mlx device: gpu" not in r.stdout:
            raise RuntimeError("the CUDA build started but MLX did not choose the GPU")
        os.environ["LD_LIBRARY_PATH"] = ld
        PHASES.append(("CUDA shears-cli (download, verify, unpack, libraries, run on GPU)", time.time() - t1))
        return cli

    cli = None
    if True:
        t0 = time.time()
        try:
            install_gxx()
            cli = try_cuda_build()
            CUDA_OK = True
            print("using the CUDA shears-cli:", cli)
        except Exception as e:
            CUDA_REASON = f"{type(e).__name__}: {e}"
            PHASES.append(("CUDA shears-cli (failed or skipped)", time.time() - t0))
            print("\n*** Not using the CUDA build:", CUDA_REASON)
            print("*** Falling back to the CPU build.\n")

    # ---------------------------------------------------------------------------------------------
    # The CPU build.
    # ---------------------------------------------------------------------------------------------
    if cli is None:
        t0 = time.time()
        try:
            cli = fetch_and_unpack(PREBUILT_URL, PREBUILT_SHA256, WORK / "prebuilt")
            # Does this binary run on this system at all (glibc, libstdc++, the bundled libraries)?
            r = subprocess.run([str(cli), "--help"], capture_output=True, text=True, timeout=120)
            if r.returncode != 0:
                raise RuntimeError(f"shears-cli --help exited {r.returncode}: {(r.stderr or r.stdout).strip()[:500]}")
            PHASES.append(("prebuilt shears-cli (download, verify, unpack, test)", time.time() - t0))
            install_gxx()
        except Exception as e:
            PHASES.append(("prebuilt shears-cli (failed)", time.time() - t0))
            raise RuntimeError(f"""Could not set up the prebuilt shears-cli: {type(e).__name__}: {e}

Nothing was built. Check the network and PREBUILT_URL in section 1, then rerun this cell.
To build shears-cli from source instead (15 to 30 minutes), set BUILD_FROM_SOURCE = True in the optional cell at the end of the notebook, run it, and then rerun from section 3.""") from None
    SHEARS_CLI = cli
    add_to_path(cli.parent)
    PREBUILT_OK = True
    print("using the prebuilt shears-cli:", cli, "(CUDA build)" if CUDA_OK else "(CPU build)")
else:
    print("not Linux: skipping")

### 2c. Which device MLX uses

`shears-cli device` runs one small matrix product on MLX's default device and prints `mlx device: gpu` or `mlx device: cpu`. **That line is the answer.** Every later `shears-cli` command prints the same line first, so it is also at the top of the output of the pipeline cells. (On a Mac, `gpu` is Metal. A CPU prebuilt that was built before this command existed answers "unknown command"; it only has the CPU backend, so the notebook says `cpu`.)

In [ ]:
import re
r = subprocess.run([str(SHEARS_CLI), "device"], capture_output=True, text=True, timeout=900)
m = re.search(r"mlx device: (\w+)", r.stdout)
if m and r.returncode == 0:
    DEVICE = m.group(1)
    print(r.stdout.strip())
elif IS_LINUX and not CUDA_OK:
    DEVICE = "cpu"
    print(f"mlx device: cpu   (this prebuilt predates `shears-cli device`, or it printed nothing; it is the CPU build, so MLX uses the CPU)")
else:
    raise RuntimeError(f"`shears-cli device` failed (exit {r.returncode}):\n{(r.stdout + r.stderr).strip()[-1500:]}")
if CUDA_REASON:
    print("(CUDA build not used:", CUDA_REASON + ")")
print(f"==> MLX is using the {'GPU' if DEVICE == 'gpu' else 'CPU'}" + (" (Metal)" if IS_MAC and DEVICE == "gpu" else " (CUDA)" if CUDA_OK else ""))

## 3. Prune and heal

**`MODEL = "tiny"`** (the cells up to section 3g): `shears-cli sample` writes a tiny random Gemma 4 MoE (`model/`: 4 layers, 8 experts each, a toy tokenizer, float32) and `rows.jsonl` (four short token rows to heal on) into a folder. `Examples/tiny-config.json` is the config for it; it names exactly those paths. The notebook cuts it to 3 heal steps per stage: this only shows the pipeline runs. Then the four commands. Each reads the same config and is safe to rerun: it resumes. With `MODEL = "gemma"` these cells print that they are skipping, and section 3g does the real thing.

In [ ]:
SETUP_MODE = "xcodebuild" if IS_MAC else (("prebuilt-cuda" if CUDA_OK else "prebuilt") if PREBUILT_OK else "source")   # source: the optional cell at the end was run
SETUP_SECONDS = sum(s for _, s in PHASES)
print(f"setup: {SETUP_MODE}, {SETUP_SECONDS:.1f} s")

In [ ]:
if MODEL == "tiny":
    TINY = WORK / "tiny"
    shutil.rmtree(TINY, ignore_errors=True)
    sh(f'"{SHEARS_CLI}" sample "{TINY}"', "sample workspace")
    # Trim the sample config to a smoke test: 3 heal steps per stage (the sample has 6, 6 and 5).
    cfg = json.load(open(SRC / "Examples/tiny-config.json"))
    for st in cfg["stages"]:
        st["steps"] = 3
    cfg["stages"][2]["plan"] = "freerun@0,end@2"   # stage C still reaches its second phase
    json.dump(cfg, open(TINY / "config.json", "w"), indent=2)
    print(sorted(p.name for p in TINY.iterdir()))
else:
    print('MODEL is gemma: skipping the tiny pipeline')

**`select`** picks 4 of the 8 experts per layer to keep, by how much dropping the others changes the layer's output on the rows.

In [ ]:
if MODEL == "tiny":
    sh(f'cd "{TINY}" && "{SHEARS_CLI}" select --config config.json', "shears-cli select")
    for i, kept in enumerate(json.load(open(TINY / "out/keep.json"))["keep"]):
        print(f"layer {i}: keeps experts {kept} of 0-7")
else:
    print('MODEL is gemma: skipping the tiny pipeline')

**`prepass`** runs the original (unpruned) model once over the rows and stores the outputs it needs, so healing never has to run the big model again. This is the "teacher".

In [ ]:
if MODEL == "tiny":
    sh(f'cd "{TINY}" && "{SHEARS_CLI}" prepass --config config.json', "shears-cli prepass")
else:
    print('MODEL is gemma: skipping the tiny pipeline')

**`heal`** trains the adapters (small add-on matrices per expert, for the dense layers, and a low-rank correction to the router) so the pruned model's outputs move back toward the teacher's. The config has three stages: A and B heal one block of layers at a time, C runs over the whole pruned model. The loss is printed per step. (A few toy steps on a random model say nothing about quality, and the loss need not fall: stage C switches between training phases (freerun, then end) that use different losses. This only shows the loop running. `mlx peak` reads 0.00 GB on the CPU backend.)

In [ ]:
if MODEL == "tiny":
    sh(f'cd "{TINY}" && "{SHEARS_CLI}" heal --config config.json', "shears-cli heal")
    for stage in "ABC":
        rows = [json.loads(l) for l in open(TINY / f"out/stage-{stage}/steps.jsonl")]
        print(f"stage {stage}: {len(rows)} steps, loss {rows[0]['loss']:.3f} -> {rows[-1]['loss']:.3f}")
else:
    print('MODEL is gemma: skipping the tiny pipeline')

**`export`** merges the healed adapters into the weights and writes only the kept experts, in Hugging Face layout (`config.json`, safetensors shards, tokenizer files), as bfloat16. The same directory feeds `shears-cli gguf` (a GGUF file for llama.cpp) and `shears-serve` (an OpenAI-style server).

In [ ]:
if MODEL == "tiny":
    sh(f'cd "{TINY}" && "{SHEARS_CLI}" export --config config.json', "shears-cli export")
    EXPORT = TINY / "out/export"
    for p in sorted(EXPORT.iterdir()):
        print(f"{p.stat().st_size:>9,}  {p.name}")
    cfg = json.load(open(EXPORT / "config.json"))
    print("\nexperts per layer in the export:", cfg["num_experts"], "(the original had 8); experts used per token:", cfg["top_k_experts"])
else:
    print('MODEL is gemma: skipping the tiny pipeline')

## 3g. The real model: `MODEL = "gemma"`

Skipped with `MODEL = "tiny"`. What this does, and what it deliberately does not:

- **Downloads `google/gemma-4-26B-A4B-it`** (Apache 2.0, not gated; about 52 GB of bf16 safetensors) after checking the disk and, if MLX ended up on the CPU, stopping first (see `ALLOW_GEMMA_ON_CPU`).
- **`shears-cli run`**, the block-sequential pipeline ([command line](https://github.com/a1cd/Shears/blob/main/docs/cli.md)): the weights are streamed one layer at a time from disk, so the 52 GB never has to fit in memory. Per block of 6 layers it chooses 8 experts to keep in each layer (of 128), then heals the block for `GEMMA_STEPS_PER_BLOCK` steps (small LoRA adapters, against the original model). The model has 5 blocks. Every block is healed because the export needs adapters for every layer and `select` alone would load the whole model; this is a short smoke-level run, not a recipe for a good model. The 32 training rows below are short chat exchanges written for this notebook; real quality needs real data and far more steps.
- **`shears-cli export`** merges the adapters and writes only the kept experts (about 3.8 billion parameters, worked out from the model's config.json, so about 7.6 GB in bf16; not measured).
- **Not checked anywhere:** that this runs to the end on a Colab GPU, how long it takes, or what the pruned model's quality is. The CUDA build has never run on a GPU.

In [ ]:
# 3g.1  Python packages, then the plan: the size of the download, the free disk, the device.
GEMMA_DIR = WORK / "gemma"
MODEL_DIR = GEMMA_DIR / "model"
if MODEL == "gemma":
    need_pkgs = [f"transformers=={TRANSFORMERS_VERSION}", "huggingface_hub"]
    try:
        import torch
    except ImportError:
        need_pkgs.append("torch")
    sh(f'"{sys.executable}" -m pip install -q --disable-pip-version-check {" ".join(need_pkgs)}', "pip install transformers")
    import transformers
    assert transformers.__version__ == TRANSFORMERS_VERSION, "restart the runtime so the pinned transformers is imported, then run again from the top"
    transformers.utils.logging.disable_progress_bar()
    from huggingface_hub import HfApi

    PATTERNS = ["*.json", "*.safetensors", "*.jinja", "tokenizer*"]
    try:
        info = HfApi().model_info(GEMMA_REPO, files_metadata=True)
        import fnmatch
        files = [s for s in info.siblings if any(fnmatch.fnmatch(s.rfilename, p) for p in PATTERNS) and not s.rfilename.startswith(".")]
        DOWNLOAD_BYTES = sum(s.size or 0 for s in files)
        size_source = "Hugging Face file list"
    except Exception as e:
        DOWNLOAD_BYTES = 51_650_000_000
        size_source = f"recorded 2026-10-22 (could not reach the file list: {type(e).__name__})"
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    have = sum(f.stat().st_size for f in MODEL_DIR.rglob("*") if f.is_file())
    remaining = max(DOWNLOAD_BYTES - have, 0)
    # After the download: the exported model (about 3.8B parameters, 7.6 GB in bf16, by calculation), the healing workspace and slack.
    WORKSPACE_BYTES = 14_000_000_000
    free = shutil.disk_usage(WORK).free
    print(f"download : {DOWNLOAD_BYTES / 1e9:.1f} GB ({size_source}); already here: {have / 1e9:.1f} GB")
    print(f"time     : at 50 to 150 MB/s that is {remaining / 150e6 / 60:.0f} to {remaining / 50e6 / 60:.0f} minutes (Colab's speed varies; not measured here)")
    print(f"disk     : {free / 1e9:.1f} GB free in {WORK}; this run needs about {(remaining + WORKSPACE_BYTES) / 1e9:.0f} GB more (download + export + workspace)")
    if free < remaining + WORKSPACE_BYTES:
        raise SystemExit(
            f"STOP: not enough disk. Need about {(remaining + WORKSPACE_BYTES) / 1e9:.0f} GB free, have {free / 1e9:.1f} GB. "
            "Delete files, pick a runtime with more disk, or set MODEL = \"tiny\" in the first code cell.")
    print(f"device   : MLX uses the {DEVICE.upper()}")
    if DEVICE == "cpu" and not IS_MAC and not ALLOW_GEMMA_ON_CPU:
        raise SystemExit(
            "STOP: MLX is on the CPU" + (f" ({CUDA_REASON})" if CUDA_REASON else "") + ". The real model on a CPU would take many hours at the very least, "
            "so the notebook stops before downloading 52 GB. On Colab pick a GPU runtime (Runtime, Change runtime type) and run again from the top. "
            "To go on anyway set ALLOW_GEMMA_ON_CPU = True in section 1, or set MODEL = \"tiny\".")
    print("ok: enough disk, and a device worth running on")
else:
    print("MODEL is tiny: skipping")

In [ ]:
# 3g.2  The config. Change the numbers here. Paths are relative to this file's folder. It is checked by `shears-cli` before anything is downloaded.
def memory_budget_gb():
    """A ceiling for `shears-cli run`: it refuses up front, with the amount, if the run needs more."""
    if os.environ.get("SHEARS_MEMORY_GB"):
        return float(os.environ["SHEARS_MEMORY_GB"])
    if IS_MAC:
        mem = int(subprocess.run(["sysctl", "-n", "hw.memsize"], capture_output=True, text=True).stdout) / 2**30
        return round(mem * 0.6, 1)
    if DEVICE == "gpu" and GPU:
        return round(GPU["mem_mib"] / 1024 * 0.85, 1)
    meminfo = dict(l.split(":", 1) for l in open("/proc/meminfo"))
    return round(int(meminfo["MemAvailable"].split()[0]) / 2**20 * 0.7, 1)

# T4 and older Turing parts have no bfloat16 arithmetic: use float32 for the maths there, and on the CPU. Layers are streamed one at a time,
# so that costs little memory. Ampere and newer, and Macs, use bfloat16. The export is bfloat16 either way.
COMPUTE_DTYPE = "float32" if (DEVICE == "cpu" and not IS_MAC) or (GPU and not IS_MAC and GPU["cc"] < 8.0) else "bfloat16"
BUDGET_GB = memory_budget_gb()
GEMMA_CONFIG = {
    "model": "model", "dtype": COMPUTE_DTYPE, "rows": "rows.jsonl", "keep": "out/keep.json",
    "order": {"kind": "shuffled", "seed": 9}, "checkpointDir": "checkpoints", "outputDir": "out", "stopFile": "STOP",
    "seed": 1, "progressEvery": 1,
    "memory": {"budgetGB": BUDGET_GB},
    "adapters": {"expertRank": 8, "denseRank": 8, "routerRank": 4},
    # 8 of 128 experts per layer. importanceCandidates and timeLimitPerLayer keep the search short; the recipe's defaults are slower.
    "select": {"keep": 8, "importanceCandidates": 32, "swapRounds": 1, "timeLimitPerLayer": 60},
    "pipeline": {"steps": GEMMA_STEPS_PER_BLOCK, "batchSteps": 5, "mix": "both", "rowsPerStep": 1, "sharedPrefix": "auto",
                 "learningRate": 0.001, "learningRateFloor": 0.0001, "warmup": 2, "checkpointEvery": 5,
                 "keepIntermediates": "lowMemory", "maxBatchTokens": 2048, "maxBatchRows": 8},
    "export": {"dtype": "bfloat16"},
}
# The export step gets the same config without "dtype": the pruned model (about 3.8B parameters) is then loaded as stored, in bfloat16 (about
# 7.6 GB), not in float32 (about 15 GB, more than a T4 or Colab's RAM holds). Training is unaffected.
EXPORT_CONFIG = {k: v for k, v in GEMMA_CONFIG.items() if k != "dtype"}
if MODEL == "gemma":
    GEMMA_DIR.mkdir(parents=True, exist_ok=True)
    json.dump(GEMMA_CONFIG, open(GEMMA_DIR / "config.json", "w"), indent=2)
    json.dump(EXPORT_CONFIG, open(GEMMA_DIR / "config-export.json", "w"), indent=2)
    print(f"compute dtype {COMPUTE_DTYPE}; memory budget {BUDGET_GB} GB (a ceiling: the run is refused up front, with the amount, if it needs more);")
    print(f"{GEMMA_STEPS_PER_BLOCK} healing steps in each of 5 blocks of 6 layers; 8 experts kept per layer of 128")
    # Does shears-cli accept the config? Run it before the 52 GB download. The model is not there yet, so it must fail, and only at that point.
    for name in ("config.json", "config-export.json"):
        cmd = "run" if name == "config.json" else "export"
        r = subprocess.run([str(SHEARS_CLI), cmd, "--config", name], cwd=GEMMA_DIR, capture_output=True, text=True, timeout=600)
        out = (r.stdout + r.stderr).strip()
        if r.returncode == 0 or any(w in out for w in ("shears-cli: config:", "DecodingError", "keyNotFound", "typeMismatch", "valueNotFound", "dataCorrupted")):
            raise RuntimeError(f"shears-cli does not accept {name} (exit {r.returncode}):\n{out[-1500:]}")
        print(f"{name}: accepted (shears-cli then stopped at the missing model, as expected: {(out.splitlines() or [''])[-1][:120]})")
else:
    print("MODEL is tiny: skipping")

In [ ]:
# 3g.3  Download the model (resumable: rerun if it stops).
if MODEL == "gemma":
    from huggingface_hub import snapshot_download
    t0 = time.time()
    snapshot_download(repo_id=GEMMA_REPO, local_dir=str(MODEL_DIR), allow_patterns=PATTERNS)
    PHASES.append(("download google/gemma-4-26B-A4B-it", time.time() - t0))
    shards = sorted(MODEL_DIR.glob("*.safetensors"))
    print(f"{len(shards)} safetensors files, {sum(p.stat().st_size for p in shards) / 1e9:.1f} GB, in {MODEL_DIR}")
    assert (MODEL_DIR / "config.json").exists() and shards, "the download is incomplete: run this cell again"
else:
    print("MODEL is tiny: skipping")

In [ ]:
# 3g.4  The training rows: 32 short chat exchanges, tokenized with the model's own chat template. answerStart marks where the reply begins.
PAIRS = [
    ("Why is the sky blue?", "Sunlight is scattered by the air. Blue light has a shorter wavelength than red, so it is scattered much more, and that scattered blue light reaches your eyes from every direction."),
    ("What is the capital of Australia?", "The capital of Australia is Canberra. Many people guess Sydney or Melbourne because those cities are larger, but Canberra was built as a compromise between them."),
    ("Explain what a hash table is.", "A hash table stores key-value pairs in an array. A hash function turns each key into an index, so a lookup usually takes the same short time however many items are stored."),
    ("Give me a tip for sleeping better.", "Keep a regular schedule: go to bed and wake up at about the same time every day, even on weekends. It helps your body clock more than almost any other single habit."),
    ("What does photosynthesis do?", "Photosynthesis lets plants turn sunlight, water and carbon dioxide into sugar and oxygen. The sugar feeds the plant, and the oxygen is released into the air."),
    ("Translate 'good morning' into Spanish.", "'Good morning' in Spanish is 'buenos dias'. You can use it until about noon."),
    ("How do I boil an egg?", "Lower the egg into boiling water and cook it for about seven minutes for a soft yolk or ten for a hard one. Then move it into cold water so it stops cooking."),
    ("What is a prime number?", "A prime number is a whole number greater than one whose only divisors are one and itself. The first few are 2, 3, 5, 7 and 11."),
    ("Summarize the plot of Romeo and Juliet.", "Two teenagers from feuding families in Verona fall in love and marry in secret. A chain of misunderstandings leads both to take their own lives, and their deaths finally end the feud."),
    ("Why do we have seasons?", "The Earth's axis is tilted. As it orbits the Sun, each hemisphere leans toward the Sun for part of the year, which gives it longer days and more direct light: summer."),
    ("What is the difference between a virus and a bacterium?", "A bacterium is a living single cell that can reproduce on its own. A virus is much smaller and can only reproduce by taking over the cells of something else."),
    ("Write a one-line greeting for a new neighbor.", "Welcome to the street! If you need anything while you settle in, knock on my door."),
    ("How does a bill become law in the United States?", "A bill passes both the House and the Senate in the same form, then goes to the President, who signs it into law or vetoes it. Congress can override a veto with a two-thirds vote."),
    ("What causes tides?", "Tides come mostly from the Moon's gravity, which pulls the ocean toward it and creates a bulge. As the Earth turns, coastlines pass through the bulges, giving two high tides a day."),
    ("Suggest a name for a bakery.", "How about 'Second Rise'? It sounds warm, hints at the craft, and it is easy to remember."),
    ("What is machine learning?", "Machine learning is a way of building software that learns patterns from examples instead of following rules written by hand. It is used for recognizing images, translating text and many other tasks."),
    ("How many days are in a leap year?", "A leap year has 366 days. The extra day, February 29, keeps the calendar in step with the Earth's orbit around the Sun."),
    ("Describe the taste of a lemon.", "A lemon is sharply sour, with a bright, slightly bitter edge from the peel. Most people find it makes their mouth water at once."),
    ("What should I pack for a day hike?", "Water, a snack, a rain layer, sun protection, a map or a downloaded route, and a small first-aid kit. Tell someone where you are going."),
    ("What is inflation?", "Inflation is a general rise in prices over time, which means each unit of money buys a little less than before. Central banks try to keep it low and steady."),
    ("Explain recursion to a beginner.", "Recursion is when a function solves a problem by calling itself on a smaller piece of the same problem. It needs a stopping case, or it would never end."),
    ("Why do cats purr?", "Cats purr when they are content, but also when they are nervous or hurt. The vibration may soothe them, and it can help kittens and their mother find each other."),
    ("What year did the Berlin Wall fall?", "The Berlin Wall fell in 1989, on the night of November 9, when East Germany opened its borders. Germany was reunified the following year."),
    ("Give me a simple recipe for tomato sauce.", "Soften a chopped onion and some garlic in olive oil, add a can of crushed tomatoes and a pinch of salt, and simmer for twenty minutes. Finish with basil."),
    ("What is the speed of light?", "Light travels at about 300,000 kilometers per second in a vacuum, which is roughly 186,000 miles per second. Nothing with mass can reach it."),
    ("How do vaccines work?", "A vaccine shows the immune system a harmless piece or weakened form of a germ. The body learns to recognize it, so it can respond fast if the real germ arrives."),
    ("What is a metaphor?", "A metaphor describes one thing as if it were another to bring out a likeness, as in 'time is a river'. It is different from a simile, which uses 'like' or 'as'."),
    ("Why is exercise good for you?", "Regular exercise strengthens the heart and muscles, improves sleep and mood, and lowers the risk of many long-term illnesses. Even a daily walk helps."),
    ("What is the largest planet in our solar system?", "Jupiter is the largest planet. It is more than eleven times wider than the Earth and has a storm, the Great Red Spot, that is larger than our planet."),
    ("How do I politely decline an invitation?", "Thank them first, say you cannot make it, and keep the reason short. For example: 'Thank you for inviting me. I can't come this time, but I hope it goes well.'"),
    ("What is the Pythagorean theorem?", "In a right triangle, the square of the longest side equals the sum of the squares of the other two sides: a squared plus b squared equals c squared."),
    ("Describe a rainy afternoon in two sentences.", "Rain tapped steadily on the window while the street outside emptied. Inside, the kettle clicked off and the whole room smelled of tea."),
]
if MODEL == "gemma":
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)

    def ids_of(messages, **kw):
        out = tok.apply_chat_template(messages, tokenize=True, return_dict=True, **kw)["input_ids"]
        return list(out[0]) if out and isinstance(out[0], (list, tuple)) else list(out)

    rows = []
    for n, (q, a) in enumerate(PAIRS):
        prompt = ids_of([{"role": "user", "content": q}], add_generation_prompt=True)
        full = ids_of([{"role": "user", "content": q}, {"role": "assistant", "content": a}])
        start = next((k for k, (x, y) in enumerate(zip(prompt, full)) if x != y), len(prompt))   # where the two renderings stop agreeing
        rows.append({"key": f"r{n:02d}", "tokens": [int(t) for t in full], "answerStart": min(start, len(full) - 1)})
    with open(GEMMA_DIR / "rows.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    lens = [len(r["tokens"]) for r in rows]
    print(f"{len(rows)} rows, {min(lens)} to {max(lens)} tokens; the first one decodes to:")
    print(repr(tok.decode(rows[0]["tokens"])[:300]))
else:
    print("MODEL is tiny: skipping")

**`run`** does the work: for each block of layers it walks the original model over the rows, chooses the experts to keep, trains the adapters, and moves on. It is resumable (rerun the cell after a stop or a disconnect: it continues from its checkpoints). **Look at its first line, `mlx device: ...`**, and at the `s/step` it reports.

In [ ]:
if MODEL == "gemma":
    sh(f'cd "{GEMMA_DIR}" && "{SHEARS_CLI}" run --config config.json', "shears-cli run (select and heal, every block)")
    keep = json.load(open(GEMMA_DIR / "out/keep.json"))["keep"]
    print(f"\nkept {len(keep[0])} experts in each of {len(keep)} layers; layer 0 keeps {keep[0]}")
    for p in sorted((GEMMA_DIR / "out").glob("**/steps.jsonl")):
        by_block = {}
        for l in open(p):
            r_ = json.loads(l)
            by_block.setdefault(r_.get("block", 0), []).append(r_["loss"])
        for b_, ls in sorted(by_block.items()):
            print(f"block {b_ + 1}: {len(ls)} steps, loss {ls[0]:.3f} -> {ls[-1]:.3f}")
else:
    print("MODEL is tiny: skipping")

**`export`** merges the adapters into the kept experts and writes the pruned model in Hugging Face layout (bfloat16).

In [ ]:
if MODEL == "gemma":
    sh(f'cd "{GEMMA_DIR}" && "{SHEARS_CLI}" export --config config-export.json', "shears-cli export")
    GEMMA_EXPORT = GEMMA_DIR / "out/export"
    for p in sorted(GEMMA_EXPORT.iterdir()):
        print(f"{p.stat().st_size:>14,}  {p.name}")
else:
    print("MODEL is tiny: skipping")

## 4. Load the export with Hugging Face transformers

The export is an ordinary Hugging Face checkpoint, so `AutoModelForCausalLM` loads it (a text-only Gemma 4, `Gemma4ForCausalLM`). transformers is pinned to a version that has Gemma 4. torch is installed too if the environment has none (Colab has one).

With `MODEL = "tiny"` it generates a few tokens of gibberish from the tiny random model. With `MODEL = "gemma"` it loads the pruned 26B (about 8 GB in bf16; on the GPU if torch can see one) and answers a real prompt. The pruned, barely healed model will answer worse than the original; that is expected.

If you are on Colab and `import transformers` ran earlier in this session, restart the runtime after the install cell so the pinned version is the one imported.

In [ ]:
need = [f"transformers=={TRANSFORMERS_VERSION}"]
try:
    import torch
except ImportError:
    need.append("torch")
sh(f'"{sys.executable}" -m pip install -q --disable-pip-version-check {" ".join(need)}', "pip install transformers")
import transformers
print("transformers", transformers.__version__)
assert transformers.__version__ == TRANSFORMERS_VERSION, "restart the runtime so the pinned transformers is imported"

In [ ]:
if MODEL == "tiny":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    transformers.utils.logging.disable_progress_bar()

    tokenizer = AutoTokenizer.from_pretrained(EXPORT)
    model = AutoModelForCausalLM.from_pretrained(EXPORT, dtype=torch.bfloat16)   # the export is bf16
    model.eval()
    print(type(model).__name__, f"| {sum(p.numel() for p in model.parameters()):,} parameters | dtype {next(model.parameters()).dtype}")

    # --- Change the prompt here. The tokenizer is a toy (lower-case letters and bytes), so use short plain text.
    messages = [{"role": "user", "content": "hello"}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)

    # --- Change sampling here. A tiny RANDOM model writes gibberish whatever you do; this only shows the model runs.
    # For varied output set do_sample=True and temperature / top_k / top_p. Greedy decoding is repeatable.
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=5, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    print("prompt   :", repr(tokenizer.decode(inputs["input_ids"][0])))
    print("generated:", repr(tokenizer.decode(new_tokens)))
    print("token ids:", new_tokens.tolist())
else:
    print('MODEL is gemma: the generation cell below uses the pruned Gemma')

In [ ]:
if MODEL == "gemma":
    import torch, gc
    from transformers import AutoTokenizer, AutoModelForCausalLM
    transformers.utils.logging.disable_progress_bar()
    gc.collect()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(GEMMA_EXPORT)

    # --- Change the prompt here.
    messages = [{"role": "user", "content": "Explain in three sentences why the sky is blue."}]

    def generate_with(dtype):
        model = AutoModelForCausalLM.from_pretrained(GEMMA_EXPORT, dtype=dtype).to(device)
        model.eval()
        print(type(model).__name__, f"| {sum(p.numel() for p in model.parameters()):,} parameters | dtype {next(model.parameters()).dtype} | on {device}")
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
        # --- Change sampling here. Greedy decoding is repeatable; for varied output set do_sample=True and temperature / top_k / top_p.
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=120, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
        return out[0, inputs["input_ids"].shape[1]:]

    t0 = time.time()
    if device == "cuda" and torch.cuda.get_device_capability()[0] < 8:
        print("note: this GPU has no bfloat16 hardware (compute capability < 8); torch emulates it, which works but is slow")
    try:
        new_tokens = generate_with(torch.bfloat16)               # the export is bf16
    except Exception as e:
        if device != "cuda":
            raise
        print(f"bfloat16 generation failed on this GPU ({type(e).__name__}: {str(e)[:200]}); trying float16. Gemma can overflow in float16, so read the answer with care.")
        gc.collect(); torch.cuda.empty_cache()
        new_tokens = generate_with(torch.float16)
    print("prompt   :", messages[0]["content"])
    print("generated:", tokenizer.decode(new_tokens, skip_special_tokens=True))
    print(f"({len(new_tokens)} tokens in {time.time() - t0:.1f} s)")
    PHASES.append(("transformers: generate from the pruned Gemma", time.time() - t0))
else:
    print("MODEL is tiny: the cell above generated from the tiny model")

## 5. How long each phase took

In [ ]:
phase_table()
print(f"setup ({SETUP_MODE}): {SETUP_SECONDS:.1f} s")
json.dump({"mode": SETUP_MODE, "setup_seconds": SETUP_SECONDS, "total_seconds": sum(s for _, s in PHASES), "phases": PHASES},
          open(WORK / "phases.json", "w"), indent=1)   # read by CI

## Where to go next

* The real thing: a Mac with Apple Silicon, `shears-cli run` on `google/gemma-4-26B-A4B-it`, with a `--memory-budget`. See the [command line page](https://github.com/a1cd/Shears/blob/main/docs/cli.md) and the [measured results](https://github.com/a1cd/Shears/blob/main/docs/benchmarks.md).
* Turn the export into a GGUF for llama.cpp: `shears-cli gguf --config config.json`, or serve it: `shears-serve serve --model <export dir> --port 8080`.
* What was verified on Linux, and what was not: [docs/linux.md](https://github.com/a1cd/Shears/blob/main/docs/linux.md).

**If the CUDA build will not start** section 2b prints the reason and uses the CPU build (please open an issue with that output: nobody has run the CUDA build on a GPU yet). **If no prebuilt binary can be set up** the notebook stops in section 2b and says why; the optional last cell builds `shears-cli` from source instead. **If a step of that build fails** the usual suspects are: `swiftly` could not install (try the tarball from swift.org by hand), the build ran out of memory (rerun the cell; MLX's C++ is memory hungry on a 2-core runtime), or a missing system library (the error names it). Open an issue with the output of the failing cell.

## 6. Optional: build `shears-cli` from source (Linux only, off by default)

**You do not need this to run the notebook.** Sections 1 to 5 use the prebuilt binary. This cell is for people changing Shears itself, or on a system the prebuilt binary does not run on (older glibc, not x86_64). It installs the Swift toolchain with `swiftly`, patches a local copy of mlx-swift (two fixes for its Linux CPU build; see [docs/linux.md](https://github.com/a1cd/Shears/blob/main/docs/linux.md)) and builds `shears-cli` from the sources in `SRC`. Expect 15 to 30 minutes on a Colab CPU runtime. On a Mac the notebook already builds from source in section 2a.

To use it: set `BUILD_FROM_SOURCE = True` below, run the cell, and then **rerun from section 3** (the pipeline cells use whatever `SHEARS_CLI` is). If section 2b stopped the notebook, run section 1 and the sources cell in section 2 first.

**EXPERIMENTAL and UNTESTED: CUDA (Linux with an NVIDIA GPU only).**

> **Nobody has run this cell.** (The prebuilt CUDA binary of section 2b is built by CI with a different recipe, in `.github/workflows/linux-cuda.yml` against CUDA 13.0; this cell is the older untested one.) It is off by default. Colab's T4 / L4 / A100 images, CUDA 13.0, and the extra headers below have never been tried with Shears, and mlx-swift's own SwiftPM CUDA build is documented as not working on its own: MLX's `.cu` sources include CUTLASS, cuDNN-frontend and CCCL headers that its CMake build downloads and its SwiftPM build does not. This cell downloads those at the versions MLX's CMake build pins and then builds a *second* `shears-cli` from a copy of the sources. If it fails, the compiler error names what is missing; the rest of the notebook still works on the CPU build above. See [docs/linux.md](https://github.com/a1cd/Shears/blob/main/docs/linux.md#read-only-cuda).
>
> To use it set `RUN_EXPERIMENTAL_CUDA = True` in section 1, then run this cell. Needs a GPU runtime with compute capability 7.5 or newer (MLX's stated minimum).

In [ ]:
# Set to True to build from source (see the text above). Default: use the prebuilt binary.
BUILD_FROM_SOURCE = False

if not BUILD_FROM_SOURCE:
    print("skipped: BUILD_FROM_SOURCE is False, the notebook uses the prebuilt shears-cli")
elif not IS_LINUX:
    print("not Linux: on a Mac the notebook builds from source in section 2a")
else:
    # ---- 1. system packages (gfortran: mlx-swift links -lgfortran; g++ stays installed for MLX's run-time kernels) ----
    sh(r"""
SUDO=""; [ "$(id -u)" = 0 ] || SUDO=sudo
export DEBIAN_FRONTEND=noninteractive
. /etc/os-release
$SUDO apt-get update -qq
$SUDO apt-get install -y -qq build-essential git curl ca-certificates python3 pkg-config binutils gnupg2 unzip tzdata \
  zlib1g-dev libncurses-dev libedit2 libsqlite3-0 libcurl4-openssl-dev libxml2-dev libz3-dev \
  libopenblas-dev liblapacke-dev gfortran ninja-build
if [ "$VERSION_ID" = "22.04" ]; then
  # Ubuntu 22.04 ships libstdc++ 12; MLX needs 13 (UNTESTED here). Take it from the toolchain PPA.
  $SUDO apt-get install -y -qq software-properties-common
  $SUDO add-apt-repository -y ppa:ubuntu-toolchain-r/test
  $SUDO apt-get update -qq
fi
$SUDO apt-get install -y -qq libstdc++-13-dev libgcc-13-dev
g++ --version | head -1
""", "apt packages")

    # ---- 2. the Swift toolchain ----
    add_to_path(pathlib.Path.home() / ".local/share/swiftly/bin")
    add_to_path(WORK / "swift-tarball/usr/bin")
    if shutil.which("swift") is None or SWIFT_VERSION not in subprocess.run(["swift", "--version"], capture_output=True, text=True).stdout:
        sh(f"""
# The verified route: swiftly. Run from a scratch directory, because `swiftly install --use` writes a .swift-version file.
mkdir -p "{WORK}/swiftly" && cd "{WORK}/swiftly"
cat > install-swiftly.sh <<'EOF'
curl -sSLO https://download.swift.org/swiftly/linux/swiftly-$(uname -m).tar.gz
tar zxf swiftly-$(uname -m).tar.gz
./swiftly init --quiet-shell-followup --assume-yes --skip-install
. ~/.local/share/swiftly/env.sh
swiftly install --use --assume-yes {SWIFT_VERSION}
EOF
if ! bash -e install-swiftly.sh; then
  echo "swiftly failed; falling back to the toolchain tarball from swift.org (UNTESTED)"
  . /etc/os-release
  P=ubuntu$(echo $VERSION_ID | tr -d .); V=ubuntu$VERSION_ID; X=""
  [ "$(uname -m)" = aarch64 ] && P=$P-aarch64 && X=-aarch64
  curl -fsSL "https://download.swift.org/swift-{SWIFT_VERSION}-release/$P/swift-{SWIFT_VERSION}-RELEASE/swift-{SWIFT_VERSION}-RELEASE-$V$X.tar.gz" -o swift.tar.gz
  mkdir -p "{WORK}/swift-tarball" && tar -xzf swift.tar.gz -C "{WORK}/swift-tarball" --strip-components=1
fi
""", "swift toolchain (swiftly)")
    sh("swift --version", "swift --version")

    # ---- 3. mlx-swift with two local fixes ----
    MLX_SWIFT = WORK / "mlx-swift"
    # The mlx-swift version Shears is pinned to.
    pins = json.load(open(SRC / "Package.resolved"))["pins"]
    mlx_version = next(p["state"]["version"] for p in pins if p["identity"] == "mlx-swift")
    if not (MLX_SWIFT / ".shears-patched").exists():
        sh(f"""
rm -rf "{MLX_SWIFT}"
git clone --quiet --depth 1 --branch {mlx_version} --recurse-submodules --shallow-submodules https://github.com/ml-explore/mlx-swift "{MLX_SWIFT}"
cd "{MLX_SWIFT}"
# 1. MLXNN needs MLXFast, which the Linux CPU manifest leaves out.
git apply "{SRC}/Notebooks/mlx-swift-linux-cpu.patch"
# 2. The shipped compiled_preamble.cpp was generated on arm64 and fails to compile elsewhere; regenerate it here.
cd Source/Cmlx && bash mlx/mlx/backend/cpu/make_compiled_preamble.sh mlx-generated/compiled_preamble.cpp g++ mlx FALSE $(uname -m)
touch "{MLX_SWIFT}/.shears-patched"
""", f"mlx-swift {mlx_version}, patched")
    else:
        print("mlx-swift already patched")

    # ---- 4. shears-cli, CPU ----
    sh(f"""
export SPM_CUDA=0
cd "{SRC}/CLI"
swift package resolve
[ -e Packages/mlx-swift ] || swift package edit mlx-swift --path "{MLX_SWIFT}"
swift build -c release --product shears-cli
""", "swift build shears-cli (release, CPU)")
    SHEARS_CLI = SRC / "CLI/.build/release/shears-cli"

    PREBUILT_OK = False

    # ---- 5. EXPERIMENTAL, UNTESTED: CUDA (RUN_EXPERIMENTAL_CUDA, see above) ----
    if RUN_EXPERIMENTAL_CUDA:
        sh(r"""
    nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv
    /usr/local/cuda/bin/nvcc --version | tail -2 || { echo 'no CUDA toolkit at /usr/local/cuda: install it first'; exit 1; }
    INC=/usr/local/cuda/include
    T=$(mktemp -d)
    [ -d $INC/cutlass ] || { git clone -q --depth 1 --branch v4.4.2 https://github.com/NVIDIA/cutlass.git $T/cutlass \
        && cp -r $T/cutlass/include/cutlass $T/cutlass/include/cute $INC/; }
    [ -f $INC/cudnn_frontend_ConvDesc.h ] || { git clone -q --depth 1 --branch v1.16.0 https://github.com/NVIDIA/cudnn-frontend.git $T/fe \
        && cp -r $T/fe/include/. $INC/; }
    if [ ! -f $INC/cccl/cuda/std/version ]; then
      curl -fsSL -o $T/cccl.zip https://github.com/NVIDIA/cccl/releases/download/v3.1.3/cccl-v3.1.3.zip
      unzip -q $T/cccl.zip -d $T/cccl && mkdir -p $INC/cccl && { cp -r $T/cccl/*/include/* $INC/cccl/ 2>/dev/null || cp -r $T/cccl/include/* $INC/cccl/; }
    fi
    """, "EXPERIMENTAL cuda headers")
        # A separate copy of the sources, so the CPU build above is left alone (the directory is still called Shears).
        CUDA_SRC = WORK / "cuda" / "Shears"
        shutil.rmtree(CUDA_SRC.parent, ignore_errors=True)
        shutil.copytree(SRC, CUDA_SRC, ignore=shutil.ignore_patterns(".build", "Packages"))
        sh(f"""
    export PATH=/usr/local/cuda/bin:$PATH LD_LIBRARY_PATH=/usr/local/cuda/lib64:${{LD_LIBRARY_PATH:-}}
    export SPM_CUDA=1 CUDA_ARCH=sm_$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
    cd "{CUDA_SRC}/CLI"
    swift package resolve
    swift package edit mlx-swift --path "{MLX_SWIFT}"
    swift build -c release --product shears-cli
    """, "EXPERIMENTAL swift build (CUDA)")
        SHEARS_CLI = CUDA_SRC / "CLI/.build/release/shears-cli"
        print("using the CUDA build:", SHEARS_CLI)

    print("built from source:", SHEARS_CLI)
